# Generating QRC inputs from Q-Chem frequency calculations

pyQRC reads a completed frequency calculation and writes a new input file whose geometry has been displaced along one or more normal modes — Silva and Goodman's *Quick Reaction Coordinate* (QRC) approach. This notebook walks through the Q-Chem example files that ship in this directory.

Requirements: `pip install pyqrc` (pulls in cclib and numpy).

> **Note:** use a cclib *release* (e.g. 1.8.1, which `pip install pyqrc` provides). cclib development builds currently have a Q-Chem parsing regression — see the README compatibility notes.


## Setup

pyQRC writes its new input files next to the file it reads, so we copy the example outputs into a `scratch/` subdirectory and run everything there. The helper below invokes the same `pyqrc` command line you would use in a terminal or HPC batch script.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()                # this examples directory
SCRATCH = HERE / "scratch"
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()

for name in ['acetaldehyde.out', 'claisen_ts.out']:
    shutil.copy(HERE / name, SCRATCH / name)


def run_pyqrc(*args):
    """Run the pyqrc command line inside the scratch directory."""
    result = subprocess.run(
        [sys.executable, "-m", "pyqrc", *args],
        cwd=SCRATCH, capture_output=True, text=True,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
        raise RuntimeError(f"pyqrc exited with code {result.returncode}")


def show(filename, n=40):
    """Print up to n lines of a file in the scratch directory."""
    lines = (SCRATCH / filename).read_text().splitlines()
    print("\n".join(lines[:n]))
    if len(lines) > n:
        print(f"... ({len(lines) - n} more lines)")


## Example 1: remove an unwanted imaginary frequency

This acetaldehyde optimization inadvertently produced a saddle point — it has one small imaginary frequency. By default pyQRC displaces along **all** imaginary modes, which is exactly what we want here: the displaced geometry breaks the symmetry of the saddle point, and re-optimizing it gives the true minimum.

In [2]:
run_pyqrc("acetaldehyde.out", "--nproc", "4", "--mem", "8GB")

o   acetaldehyde.out had 1 imaginary frequencies: processed


That wrote two files: `acetaldehyde_QRC.inp` (the new, displaced input — ready to submit) and `acetaldehyde_QRC.qrc` (a human-readable summary of the frequencies and the displacement).

In [3]:
show("acetaldehyde_QRC.inp")

$molecule
0 1
 C  -0.23989603  -0.41400014  -0.01839886
 O  -1.21878387   0.28856908   0.01499978
 H  -0.34220289  -1.51972004  -0.08160123
 C   1.16775303   0.13590817   0.00500103
 H   1.91671284  -0.65642526   0.10346628
 H   1.34472973   0.68481398  -0.93074441
 H   1.26388921   0.85133056   0.83146815
$end

$rem
   JOBTYPE opt
   METHOD M062X
   BASIS 6-31G(d,p)
$end

@@@

$molecule
   read
$end

$rem
   JOBTYPE freq
   METHOD M062X
   BASIS 6-31G(d,p)
$end



In [4]:
show("acetaldehyde_QRC.qrc", n=24)

 pyQRC - a quick alternative to IRC calculations
 version: 2.3.0 / author: Robert Paton / email: robert.paton@colostate.edu
 Based on: Goodman, J. M.; Silva, M. A. Tet. Lett. 2003, 44, 8233-8236;
 Tet. Lett. 2005, 46, 2067-2069.

                -----ORIGINAL GEOMETRY------
                       X         Y         Z
   C           -0.239896 -0.414000  0.000001
   O           -1.218784  0.288569 -0.000000
   H           -0.342203 -1.519720 -0.000001
   C            1.167753  0.135908  0.000001
   H            1.916713 -0.656425  0.000066
   H            1.304330  0.768014 -0.881144
   H            1.304289  0.768131  0.881068

                ----HARMONIC FREQUENCIES----
                    Freq  Red mass   F const
               -169.8000    1.1906    0.0202
                507.7000    2.6427    0.4013
                754.9600    1.1713    0.3934
                948.6000    2.1552    1.1426
               1107.9600    2.0111    1.4545
               1140.6500    1.7254    1.3226
    

## Example 2: map a reaction coordinate (the namesake QRC)

For a transition state — here a Claisen rearrangement — the quick alternative to an IRC is two displaced inputs: one along the imaginary mode (`--amp 0.3`) and one in the reverse direction (`--amp -0.3`). Optimizing both gives the reactant and product the TS connects. The benchmark in the README found an amplitude of **0.3** performs best, hence the values used here; `--name` controls the suffix of the generated files.

In [5]:
run_pyqrc("claisen_ts.out", "--nproc", "4", "--mem", "8GB", "--amp", "0.3", "--name", "QRCF")
run_pyqrc("claisen_ts.out", "--nproc", "4", "--mem", "8GB", "--amp", "-0.3", "--name", "QRCR")

o   claisen_ts.out had 1 imaginary frequencies: processed


o   claisen_ts.out had 1 imaginary frequencies: processed


In [6]:
show("claisen_ts_QRCF.inp", n=14)
print("=" * 60)
show("claisen_ts_QRCR.inp", n=14)

$molecule
0 1
 C   1.57725331  -0.64016682  -0.31479468
 C   1.24570603   0.51979610   0.25798469
 O   0.29520089   1.34820085  -0.23764386
 C  -1.17384866   0.87557915   0.16086163
 C  -1.32123400  -0.51574983  -0.29716709
 C  -0.65796743  -1.47825339   0.34671231
 H   2.27854629  -1.31766805   0.16438926
 H   1.22653751  -0.90042035  -1.31761742
 H   1.60308445   0.76370424   1.26300924
 H  -1.78054359   1.61880696  -0.35396660
 H  -1.24016236   0.99745295   1.25640546
 H  -1.63809993  -0.67009255  -1.32561129
... (22 more lines)
$molecule
0 1
 C   1.37145331  -0.76136682  -0.26859468
 C   1.23910603   0.57139610   0.25198469
 O   0.51240089   1.40820085  -0.27064386
 C  -1.45824866   0.76817915   0.20586163
 C  -1.33803400  -0.47494983  -0.29776709
 C  -0.42216743  -1.41225339   0.30471231
 H   2.11234629  -1.41246805   0.19498926
 H   1.28293751  -0.86142035  -1.35901742
 H   1.61868445   0.73490424   1.28160924
 H  -1.92994359   1.57740696  -0.34316660
 H  -1.19456236   0.99385295

## Using the Python API instead of the CLI

The same machinery is importable. `QRCGenerator` parses the output, computes the displaced geometry, and (unless `write=False`) writes the files in one go. With `write=False` you can inspect the displacement before committing anything to disk.

In [7]:
import numpy as np
from pyqrc import QRCGenerator

qrc = QRCGenerator(
    file=str(SCRATCH / "claisen_ts.out"),
    amplitude=0.3,
    nproc=4,
    mem="8GB",
    route=None,     # None clones the route/keywords from the original job
    verbose=False,  # skip the .qrc summary file
    suffix="API",
    val=None,       # or displace along the mode nearest this frequency (cm-1)
    num=None,       # or along this 1-indexed mode number
    write=False,    # compute only; no files are written
)

freqs = np.asarray(qrc.FREQS)
print("Imaginary frequencies (cm-1):", freqs[freqs < 0.0])
print(f"Mass-weighted displacement from the TS: {qrc.MW_DISTANCE:.4f} bohr amu^1/2")
print("Displaced geometry has clashing atoms:", qrc.OVERLAPPED)
print()

per_atom = np.linalg.norm(qrc.NEW_CARTESIAN - qrc.CARTESIAN, axis=1)
print("Atoms that move the most:")
for i in np.argsort(per_atom)[::-1][:5]:
    print(f"  atom {i + 1:>2} ({qrc.ATOMTYPES[i]:<2}) moved {per_atom[i]:.3f} Angstrom")


Imaginary frequencies (cm-1): [-588.44]
Mass-weighted displacement from the TS: 1.7830 bohr amu^1/2
Displaced geometry has clashing atoms: False

Atoms that move the most:
  atom  4 (C ) moved 0.154 Angstrom
  atom  6 (C ) moved 0.124 Angstrom
  atom  1 (C ) moved 0.122 Angstrom
  atom  3 (O ) moved 0.114 Angstrom
  atom  7 (H ) moved 0.097 Angstrom


## Where the files went

Everything generated above is in the `scratch/` subdirectory (ignored by git) — delete it when you are done. In real use you would submit the new input file (`*.inp`) to your scheduler and optimize.

See the [project README](../../README.md) for the full option list and for the IRC-comparison benchmark behind the recommended amplitude of 0.3.
